In [1]:
!pip -q install openai sympy

import re
import time
from google.colab import userdata
OPENAI_API_KEY = userdata.get('OA_TOKEN')

from openai import OpenAI
client = OpenAI(api_key=OPENAI_API_KEY)


In [14]:
from datetime import datetime
def make_log_filename(n, i):
    return f"failure_n{n}_i{i}.txt"


In [3]:
MODEL = "gpt-4.1-nano"   # or "gpt-4.1-nano"

TEMPERATURE = 1.0        # try 0.0 → 1.0
TOP_P = 1.0              # try 0.1 → 1.0


In [4]:
def gpt_multiply(a, b):
    prompt = f"""
Multiply the following numbers.

Return ONLY the final integer.
No explanation.
No extra text.

{a} * {b}
"""
    response = client.responses.create(
        model=MODEL,
        input=prompt,
        temperature=TEMPERATURE,
        top_p=TOP_P
    )

    return response.output_text.strip()


In [5]:
def ask_ai(expr: str, model="gpt-4.1-mini", temperature=0.0, top_p=0.2):
    prompt = f"""
You are a math calculator.

Solve the user's expression exactly if possible.
Return ONLY the final answer.
- No explanation
- No steps
- No extra text
- If the result is irrational, you may return a simplified exact form (like sqrt(2)) or a decimal.
Expression:
{expr}
"""
    resp = client.responses.create(
        model=model,
        input=prompt,
        temperature=temperature,
        top_p=top_p,
    )
    return resp.output_text.strip()


In [6]:
def extract_int(text):
    match = re.search(r"-?\d+", text)
    return int(match.group()) if match else None


In [7]:
SELF_DEPRECATING_LINES = [
    "Wow, I messed up again. Math is hard, okay?",
    "I swear I knew this one.",
    "This is deeply embarrassing for an AI.",
    "I was trained on billions of words and still can't multiply.",
    "Please stop giving me numbers.",
    "At this point I'm just guessing.",
    "I miss calculators.",
    "I should not have skipped math class."
]


In [13]:
def run_bot():
    print("AI Repeated Multiplication Bot")
    print("-----------------------------------")

    n = int(input("Enter base number n: "))
    i = int(input("Enter number of iterations i: "))

    log_file = make_log_filename(n, i)

    with open(log_file, "w", encoding="utf-8") as f:
        f.write("AI Repeated Multiplication Bot — Failure Log\n")
        f.write("=" * 60 + "\n")
        f.write(f"Run date: {datetime.now()}\n")
        f.write(f"Parameters: n = {n}, i = {i}\n")
        f.write(f"Model: {MODEL}\n")
        f.write(f"Temperature: {TEMPERATURE}\n")
        f.write(f"top_p: {TOP_P}\n")
        f.write("=" * 60 + "\n\n")

    current = n
    error_count = 0
    first_error_step = None

    PREVIEW_STEPS = 3
    TAIL_STEPS = 3

    for step in range(1, i + 1):

        correct = current * current
        gpt_text = gpt_multiply(current, current)
        gpt_value = extract_int(gpt_text)

        is_correct = (gpt_value == correct)

        if not is_correct:
            error_count += 1
            if first_error_step is None:
                first_error_step = step

        should_print = (
            step <= PREVIEW_STEPS or
            step > i - TAIL_STEPS
        )

        if should_print:
            print(f"Step {step}: {current} * {current}")

            if is_correct:
                print("→ GPT:", gpt_text, "✅")
            else:
                idx = min(error_count - 1, len(SELF_DEPRECATING_LINES) - 1)
                line = SELF_DEPRECATING_LINES[idx]
                print("→ GPT:", gpt_text, "❌")
                print("🤖", line)

                with open(log_file, "a", encoding="utf-8") as f:
                    f.write(f"Step {step}: INCORRECT\n")
                    f.write(f"GPT output: {gpt_text}\n")
                    f.write("Correct result: (very large number omitted)\n")
                    f.write(f"Bot response: {line}\n\n")

            print("-----------------------------------")

        if step == PREVIEW_STEPS + 1 and i > PREVIEW_STEPS + TAIL_STEPS:
            skipped = i - (PREVIEW_STEPS + TAIL_STEPS)
            print(f"... skipping {skipped} iterations ...\n")

        current = correct

    print("\nFinished.")
    print("Total iterations:", i)
    print("Total errors:", error_count)
    print("Log file saved as:", log_file)

    with open(log_file, "a", encoding="utf-8") as f:
        f.write("-" * 60 + "\n")
        f.write(f"First error at iteration: {first_error_step}\n")
        f.write(f"Total errors: {error_count}\n")


In [19]:
run_bot()


AI Repeated Multiplication Bot
-----------------------------------
Enter base number n: 12
Enter number of iterations i: 5
Step 1: 12 * 12
→ GPT: 144 ✅
-----------------------------------
Step 2: 144 * 144
→ GPT: 20736 ✅
-----------------------------------
Step 3: 20736 * 20736
→ GPT: 43046721 ❌
🤖 Wow, I messed up again. Math is hard, okay?
-----------------------------------
Step 4: 429981696 * 429981696
→ GPT: 184661212102762496 ❌
🤖 I swear I knew this one.
-----------------------------------
Step 5: 184884258895036416 * 184884258895036416
→ GPT: 34251461799332259920932499367519263783643260805657447616 ❌
🤖 This is deeply embarrassing for an AI.
-----------------------------------

Finished.
Total iterations: 5
Total errors: 3
Log file saved as: failure_n12_i5.txt
